### Imports & Configrations

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import DataFrame
import re
from datetime import datetime


BRONZE_FILES = {
    "customers":       "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/2f855148-2e3c-4ca1-b8a3-6d49d05e642d/Files/customers_full_final_txt",
    "orders":          "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/2f855148-2e3c-4ca1-b8a3-6d49d05e642d/Files/orders_txt",
    "order_items":     "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/2f855148-2e3c-4ca1-b8a3-6d49d05e642d/Files/order_items_txt",
    "products":        "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/2f855148-2e3c-4ca1-b8a3-6d49d05e642d/Files/products_txt",
    "product_reviews": "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/2f855148-2e3c-4ca1-b8a3-6d49d05e642d/Files/product_reviews_txt"
}

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

### Helper Functions

In [ ]:

def standardize_columns(df: DataFrame) -> DataFrame:
    """Lowercase, trim, replace spaces with underscores"""
    for col_name in df.columns:
        new_name = re.sub(r'[^a-z0-9_]', '', 
                          col_name.strip().lower().replace(' ', '_'))
        df = df.withColumnRenamed(col_name, new_name)
    return df


def read_bronze_txt(path: str) -> DataFrame:
    """Read TXT from Bronze"""
    df = (spark.read
        .option("header", "true")
        .option("delimiter", "\t")
        .option("inferSchema", "false")
        .csv(path)
    )
    return standardize_columns(df)


def write_to_silver(df: DataFrame, table_name: str):
    """ Write to Silver """
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)
    print(f"    Written to Silver: {table_name} ({df.count()} rows)")


StatementMeta(, , -1, Cancelled, , Cancelled, True)

### Customers File

In [ ]:
df_raw = read_bronze_txt(BRONZE_FILES["customers"])
print(f"Number of rows before cleaning: {df_raw.count()}")

# TYPE CASTING 
df = (df_raw
    .withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("gender", F.initcap(F.trim(F.col("gender"))))
    .withColumn("start_date", F.to_date(F.col("start_date"), "M/d/yyyy"))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("status", F.col("end_date").cast(IntegerType())) 
    .withColumn("end_date", F.lit(None).cast(DateType()))         
)

# STANDARDIZE GENDER
valid_genders = ["Male", "Female", "Other"]
df = df.withColumn(
    "gender",
    F.when(F.col("gender").isin(valid_genders), F.col("gender"))
     .otherwise(F.lit("Unknown"))
)

# VALIDATE & DROP BAD ROWS
df = df.filter(
    (F.col("customer_id").isNotNull()) &
    (F.col("name").isNotNull()) & (F.length(F.col("name")) > 0) &
    (F.col("email").isNotNull()) &
    (F.col("email").rlike(r"^[\w\.\-\+]+@[\w\.\-]+\.\w{2,}$")) &
    (F.col("start_date").isNotNull()) &
    (F.col("status").isin(0, 1))
)

# DEDUP & WRITE
df = df.dropDuplicates(["customer_id"])
write_to_silver(df, "customers")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

### Orders File

In [ ]:
df_raw = read_bronze_txt(BRONZE_FILES['orders'])
print(f"Number of rows before cleaning: {df_raw.count()}")

# TYPE CASTING 
df = (df_raw
    .withColumn("order_id", F.col("order_id").cast(IntegerType()))
    .withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd"))
    .withColumn("total_amount", F.col("total_amount").cast(DecimalType(12, 2)))
    .withColumn("payment_method", F.trim(F.col("payment_method")))
    .withColumn("shipping_country", F.trim(F.col("shipping_country")))
)

# STANDARDIZE PAYMENT METHOD
valid_payments = ["Credit Card", "Bank Transfer", "PayPal", "Cash", "Debit Card"]
df = df.withColumn(
    "payment_method",
    F.when(F.col("payment_method").isin(valid_payments), F.col("payment_method"))
     .otherwise(F.lit("Unknown"))
)

# FLAG ZERO-AMOUNT ORDERS 
df = df.withColumn(
    "is_zero_amount",
    F.when(F.col("total_amount") == 0, F.lit(True)).otherwise(F.lit(False))
)

# VALIDATE & DROP BAD ROWS
df = df.filter(
    (F.col("order_id").isNotNull()) &
    (F.col("customer_id").isNotNull()) &
    (F.col("order_date").isNotNull()) &
    (F.col("total_amount").isNotNull()) & (F.col("total_amount") >= 0) &
    (F.col("payment_method").isNotNull()) &
    (F.col("shipping_country").isNotNull()) & (F.length(F.col("shipping_country")) > 0)
)

# DEDUP & WRITE
df = df.dropDuplicates(["order_id"])
write_to_silver(df, "orders")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

### Order Items File

In [ ]:
df_raw = read_bronze_txt(BRONZE_FILES['order_items'])
print(f"Number of rows before cleaning: {df_raw.count()}")

# TYPE CASTING
df = (df_raw
    .withColumn("order_item_id", F.col("order_item_id").cast(IntegerType()))
    .withColumn("order_id", F.col("order_id").cast(IntegerType()))
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price", F.col("unit_price").cast(DecimalType(10, 2)))
)

#  DERIVED COLUMN
df = df.withColumn(
    "line_total",
    F.round(F.col("quantity") * F.col("unit_price"), 2).cast(DecimalType(12, 2))
)

# VALIDATE & DROP BAD ROWS
df = df.filter(
    (F.col("order_item_id").isNotNull()) &
    (F.col("order_id").isNotNull()) &
    (F.col("product_id").isNotNull()) &
    (F.col("quantity").isNotNull()) & (F.col("quantity") > 0) &
    (F.col("unit_price").isNotNull()) & (F.col("unit_price") >= 0)
)

# DEDUP & WRITE
df = df.dropDuplicates(["order_item_id"])
write_to_silver(df, "order_items")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

### Products File

In [ ]:
df_raw = read_bronze_txt(BRONZE_FILES['products'])
print(f"Number of rows before cleaning: {df_raw.count()}")

# TYPE CASTING
df = (df_raw
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("price", F.col("price").cast(DecimalType(10, 2)))
    .withColumn("stock_quantity", F.col("stock_quantity").cast(IntegerType()))
    .withColumn("brand", F.trim(F.col("brand")))
)

# STANDARDIZE CATEGORY
valid_categories = ["Books", "Beauty", "Toys", "Electronics", "Clothing", "Home", "Sports"]
df = df.withColumn(
    "category",
    F.when(F.col("category").isin(valid_categories), F.col("category"))
     .otherwise(F.lit("Other"))
)

# STOCK STATUS FLAG
df = df.withColumn(
    "stock_status",
    F.when(F.col("stock_quantity") == 0, F.lit("Out of Stock"))
     .when(F.col("stock_quantity") < 10, F.lit("Low Stock"))
     .when(F.col("stock_quantity") < 100, F.lit("Medium Stock"))
     .otherwise(F.lit("In Stock"))
)

# VALIDATE & DROP BAD ROWS
df = df.filter(
    (F.col("product_id").isNotNull()) &
    (F.col("product_name").isNotNull()) & (F.length(F.col("product_name")) > 0) &
    (F.col("price").isNotNull()) & (F.col("price") >= 0) &
    (F.col("stock_quantity").isNotNull()) & (F.col("stock_quantity") >= 0) &
    (F.col("brand").isNotNull()) & (F.length(F.col("brand")) > 0)
)

# DEDUP & WRITE
df = df.dropDuplicates(["product_id"])
write_to_silver(df, "products")

### Product Reviews File

In [ ]:
df_raw = read_bronze_txt(BRONZE_FILES['product_reviews'])
print(f"Number of rows before cleaning: {df_raw.count()}")

# TYPE CASTING
df = (df_raw
    .withColumn("review_id", F.col("review_id").cast(IntegerType()))
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("rating", F.col("rating").cast(IntegerType()))
    .withColumn("review_text", F.trim(F.col("review_text")))
    .withColumn("review_date", F.to_date(F.col("review_date"), "yyyy-MM-dd"))
)

# RATING LABEL
df = df.withColumn(
    "rating_label",
    F.when(F.col("rating") >= 4, F.lit("Positive"))
     .when(F.col("rating") == 3, F.lit("Neutral"))
     .otherwise(F.lit("Negative"))
)

# FUTURE DATE FLAG
df = df.withColumn(
    "is_future_date",
    F.when(F.col("review_date") > F.current_date(), F.lit(True))
     .otherwise(F.lit(False))
)

# VALIDATE & DROP BAD ROWS
df = df.filter(
    (F.col("review_id").isNotNull()) &
    (F.col("product_id").isNotNull()) &
    (F.col("customer_id").isNotNull()) &
    (F.col("rating").isNotNull()) & (F.col("rating").between(1, 5)) &
    (F.col("review_date").isNotNull()) &
    (F.col("review_text").isNotNull()) & (F.length(F.col("review_text")) > 0)
)

# DEDUP & WRITE
df = df.dropDuplicates(["review_id"])
write_to_silver(df, "product_reviews")

### Final Validation

In [8]:
print("SILVER LAYER VERIFICATION:")


for table in ["customers", "orders", "order_items", "products", "product_reviews"]:
    try:
        df = spark.read.format("delta").table(table)
        print(f"\n {table.upper()} — {df.count()} rows, {len(df.columns)} cols")
        for field in df.schema.fields:
            nullable = "nullable" if field.nullable else "not null"
            print(f"   ├── {field.name:<25} {str(field.dataType):<20} {nullable}")
    except Exception as e:
        print(f"\n {table.upper()} — ERROR: {str(e)}")

print("\n DONE!")

StatementMeta(, f1655c0f-b33d-47cd-924b-5bccf20f65c1, 10, Finished, Available, Finished, True)

SILVER LAYER VERIFICATION:

 CUSTOMERS — 1048575 rows, 8 cols
   ├── customer_id               IntegerType()        nullable
   ├── name                      StringType()         nullable
   ├── email                     StringType()         nullable
   ├── gender                    StringType()         nullable
   ├── start_date                DateType()           nullable
   ├── country                   StringType()         nullable
   ├── end_date                  DateType()           nullable
   ├── status                    IntegerType()        nullable

 ORDERS — 8000000 rows, 7 cols
   ├── order_id                  IntegerType()        nullable
   ├── customer_id               IntegerType()        nullable
   ├── order_date                DateType()           nullable
   ├── total_amount              DecimalType(12,2)    nullable
   ├── payment_method            StringType()         nullable
   ├── shipping_country          StringType()         nullable
   ├── is_zero_amount   